<a href="https://colab.research.google.com/github/aidev-ahmedamr/real-time-dynamic-pricing-engine/blob/main/notebooks/01_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

In [2]:
categories = {
    "Electronics": {
        "products": [
            "Wireless Headphones",
            "Smartphone",
            "Laptop",
            "Smart Watch",
            "Bluetooth Speaker"
        ],
        "base_price_range": (50, 1500),
        "elasticity": 1.3
    },

    "Fashion": {
        "products": [
            "Running Shoes",
            "T-Shirt",
            "Jacket",
            "Jeans",
            "Backpack"
        ],
        "base_price_range": (20, 250),
        "elasticity": 1.8
    },

    "Home & Kitchen": {
        "products": [
            "Coffee Maker",
            "Blender",
            "Air Fryer",
            "Vacuum Cleaner",
            "Microwave"
        ],
        "base_price_range": (30, 500),
        "elasticity": 1.4
    },

    "Sports": {
        "products": [
            "Dumbbells",
            "Yoga Mat",
            "Football",
            "Treadmill",
            "Fitness Tracker"
        ],
        "base_price_range": (15, 1000),
        "elasticity": 1.5
    },

    "Beauty": {
        "products": [
            "Face Cream",
            "Perfume",
            "Hair Dryer",
            "Skin Care Set",
            "Electric Shaver"
        ],
        "base_price_range": (10, 300),
        "elasticity": 1.6
    }
}

In [3]:
products = []

product_id = 1

for category, info in categories.items():
    for product_name in info["products"]:

        base_price = round(
            np.random.uniform(
                info["base_price_range"][0],
                info["base_price_range"][1]
            ),
            2
        )

        cost_price = round(
            base_price * np.random.uniform(0.4, 0.75),
            2
        )

        products.append({
            "product_id": f"P{product_id:04d}",
            "product_name": product_name,
            "category": category,
            "base_price": base_price,
            "cost_price": cost_price,
            "price_elasticity": info["elasticity"],
            "initial_inventory": np.random.randint(50, 500)
        })

        product_id += 1

products_df = pd.DataFrame(products)

products_df.head()

,product_id,product_name,category,base_price,cost_price,price_elasticity,initial_inventory
0,P0001,Wireless Headphones,Electronics,593.08,434.58,1.3,156
1,P0002,Smartphone,Electronics,1180.55,718.83,1.3,171
2,P0003,Laptop,Electronics,276.19,116.09,1.3,137
3,P0004,Smart Watch,Electronics,533.88,240.25,1.3,180
4,P0005,Bluetooth Speaker,Electronics,79.85,59.05,1.3,463


In [4]:
expanded_products = []

product_id = 1

for category, info in categories.items():

    for i in range(100):

        base_price = round(
            np.random.uniform(
                info["base_price_range"][0],
                info["base_price_range"][1]
            ),
            2
        )

        cost_price = round(
            base_price * np.random.uniform(0.4, 0.75),
            2
        )

        expanded_products.append({
            "product_id": f"P{product_id:05d}",
            "product_name": f"{category}_Product_{i+1}",
            "category": category,
            "base_price": base_price,
            "cost_price": cost_price,
            "price_elasticity": info["elasticity"],
            "initial_inventory": np.random.randint(50, 500),
            "rating": round(np.random.uniform(3.0, 5.0), 2),
            "num_reviews": np.random.randint(10, 5000)
        })

        product_id += 1

products_df = pd.DataFrame(expanded_products)

products_df.shape

(500, 9)

In [5]:
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

date_range = pd.date_range(
    start=start_date,
    end=end_date,
    freq="H"
)

len(date_range)

/tmp/ipykernel_566/2930729340.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range = pd.date_range(


8737

In [6]:
def generate_record(product, timestamp):

    hour = timestamp.hour
    day_of_week = timestamp.dayofweek
    month = timestamp.month

    is_weekend = int(day_of_week >= 5)

    seasonal_factor = 1.0

    if month in [11, 12]:
        seasonal_factor = 1.3

    elif month in [6, 7, 8]:
        seasonal_factor = 1.1

    traffic_factor = 1.0

    if 18 <= hour <= 23:
        traffic_factor = 1.5

    elif 8 <= hour <= 10:
        traffic_factor = 1.2

    competitor_price = product["base_price"] * np.random.uniform(0.85, 1.15)

    current_price = product["base_price"] * np.random.uniform(0.75, 1.25)

    inventory_level = np.random.randint(5, product["initial_inventory"])

    views_last_hour = int(
        np.random.poisson(50 * traffic_factor * seasonal_factor)
    )

    add_to_cart_count = int(
        views_last_hour * np.random.uniform(0.05, 0.25)
    )

    price_ratio = current_price / product["base_price"]

    price_effect = price_ratio ** (-product["price_elasticity"])

    competitor_effect = (
        competitor_price / current_price
    ) ** 0.8

    inventory_factor = min(
        1.2,
        max(0.3, inventory_level / 100)
    )

    base_demand = 10

    demand = (
        base_demand
        * price_effect
        * competitor_effect
        * traffic_factor
        * seasonal_factor
        * inventory_factor
        * np.random.uniform(0.7, 1.3)
    )

    demand = max(0, int(demand))

    conversion_rate = demand / max(views_last_hour, 1)

    return {
        "timestamp": timestamp,
        "product_id": product["product_id"],
        "category": product["category"],

        "base_price": product["base_price"],
        "cost_price": product["cost_price"],
        "current_price": round(current_price, 2),
        "competitor_price": round(competitor_price, 2),

        "inventory_level": inventory_level,

        "views_last_hour": views_last_hour,
        "add_to_cart_count": add_to_cart_count,

        "hour": hour,
        "day_of_week": day_of_week,
        "month": month,

        "is_weekend": is_weekend,

        "rating": product["rating"],
        "num_reviews": product["num_reviews"],

        "conversion_rate": round(conversion_rate, 4),

        "demand": demand
    }

In [7]:
records = []

sample_products = products_df.sample(
    n=300,
    random_state=42
).to_dict("records")

sample_dates = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="6H"
)

for product in sample_products:
    for timestamp in sample_dates:

        record = generate_record(
            product,
            timestamp
        )

        records.append(record)

data = pd.DataFrame(records)

data.shape

/tmp/ipykernel_566/3671666078.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  sample_dates = pd.date_range(


(437100, 18)

In [8]:
data.head()

,timestamp,product_id,category,base_price,cost_price,current_price,competitor_price,inventory_level,views_last_hour,add_to_cart_count,hour,day_of_week,month,is_weekend,rating,num_reviews,conversion_rate,demand
0,2025-01-01 00:00:00,P00362,Sports,514.24,234.37,630.08,537.72,166,45,10,0,2,1,0,3.24,1611,0.1111,5
1,2025-01-01 06:00:00,P00362,Sports,514.24,234.37,541.00,443.24,31,41,10,6,2,1,0,3.24,1611,0.0488,2
2,2025-01-01 12:00:00,P00362,Sports,514.24,234.37,436.10,511.47,39,49,8,12,2,1,0,3.24,1611,0.1020,5
3,2025-01-01 18:00:00,P00362,Sports,514.24,234.37,398.41,568.87,140,52,6,18,2,1,0,3.24,1611,0.7885,41
4,2025-01-02 00:00:00,P00362,Sports,514.24,234.37,517.50,525.36,111,33,7,0,3,1,0,3.24,1611,0.2727,9


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 437100 entries, 0 to 437099
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   timestamp          437100 non-null  datetime64[ns]
 1   product_id         437100 non-null  object        
 2   category           437100 non-null  object        
 3   base_price         437100 non-null  float64       
 4   cost_price         437100 non-null  float64       
 5   current_price      437100 non-null  float64       
 6   competitor_price   437100 non-null  float64       
 7   inventory_level    437100 non-null  int64         
 8   views_last_hour    437100 non-null  int64         
 9   add_to_cart_count  437100 non-null  int64         
 10  hour               437100 non-null  int64         
 11  day_of_week        437100 non-null  int64         
 12  month              437100 non-null  int64         
 13  is_weekend         437100 non-null  int64   

In [10]:
data.describe()

,timestamp,base_price,cost_price,current_price,competitor_price,inventory_level,views_last_hour,add_to_cart_count,hour,day_of_week,month,is_weekend,rating,num_reviews,conversion_rate,demand
count,437100,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000,437100.000000
mean,2025-07-02 00:00:00.000000256,353.818633,205.442967,353.888255,353.808140,138.522478,60.481741,8.567639,8.993823,2.999314,6.514756,0.285518,4.049700,2546.306667,0.185999,11.071354
min,2025-01-01 00:00:00,12.040000,5.060000,9.030000,10.240000,5.000000,22.000000,1.000000,0.000000,0.000000,1.000000,0.000000,3.000000,12.000000,0.010500,1.000000
25%,2025-04-02 00:00:00,132.730000,72.102500,125.850000,130.260000,51.000000,49.000000,5.000000,0.000000,1.000000,4.000000,0.000000,3.590000,1305.750000,0.092600,5.000000
50%,2025-07-02 00:00:00,226.620000,125.875000,224.540000,225.400000,110.000000,57.000000,8.000000,6.000000,3.000000,7.000000,0.000000,4.105000,2673.500000,0.163300,9.000000
75%,2025-10-01 00:00:00,440.545000,268.802500,461.445000,453.942500,204.000000,69.000000,11.000000,12.000000,5.000000,10.000000,1.000000,4.582500,3758.000000,0.250000,15.000000
max,2025-12-31 00:00:00,1494.100000,1016.220000,1867.580000,1717.380000,498.000000,149.000000,31.000000,18.000000,6.000000,12.000000,1.000000,4.990000,4958.000000,1.185200,66.000000
std,NaN,331.127558,203.781122,338.442038,333.691499,107.955496,15.321127,4.281465,6.710050,1.999487,3.442435,0.451661,0.578142,1419.228523,0.120543,7.530749


In [14]:
data.shape

(437100, 18)

In [16]:
import os

os.path.exists("dynamic_pricing_raw.csv")

True

In [18]:
records = []

sample_products = products_df.sample(
    n=100,
    random_state=42
).to_dict("records")

sample_dates = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="12H"
)

for product in sample_products:
    for timestamp in sample_dates:
        records.append(
            generate_record(product, timestamp)
        )

data = pd.DataFrame(records)

print("Shape:", data.shape)
print("Memory usage:", round(data.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

/tmp/ipykernel_566/1758759067.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  sample_dates = pd.date_range(


Shape: (72900, 18)
Memory usage: 16.71 MB


In [19]:
data.to_csv(
    "dynamic_pricing_raw.csv",
    index=False
)

In [20]:
import os

file_size = os.path.getsize("dynamic_pricing_raw.csv") / (1024 * 1024)

print(f"File size: {file_size:.2f} MB")

File size: 6.94 MB


In [21]:
from google.colab import files

files.download("dynamic_pricing_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>